In [3]:
!pip install accelerate==1.10.0
!pip install datasets==4.0.0
!pip install peft==0.17.0
!pip install transformers==4.55.2
!pip install trl==0.21.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 27.3 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.10.1
    Uninstalling accelerate-1.10.1:
      Successfully uninstalled accelerate-1.10.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 34.3 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.17.1
    Uninstalling peft-0.17.1:
      Successfully uninstalled peft-0.17.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 160.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 128.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling tran

In [4]:
import accelerate
import datasets
import peft
import transformers
import trl

print("Library Versions:")
print(f"accelerate   : {accelerate.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"peft         : {peft.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")

Library Versions:
accelerate   : 1.10.0
datasets     : 4.0.0
peft         : 0.17.0
transformers : 4.55.2
trl          : 0.21.0


In [5]:
# Verify PyTorch and CUDA versions for compatibility with the training setup.
# Used in latest run.
# PyTorch 2.8.0+cu126 built against CUDA 12.6.
# NVIDIA A100 (Driver 550.54.15, CUDA 12.4 runtime).
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Version:", torch.version.cuda)

PyTorch Version: 2.8.0+cu126
CUDA Version: 12.6


In [6]:
# Check the number of GPUs available
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus}")

# Check if CUDA device 1 is available
if num_gpus > 1:
    print("cuda:1 is available.")
else:
    print("cuda:1 is not available.")

Number of GPUs available: 1
cuda:1 is not available.


In [7]:
!nvidia-smi

Tue Oct 21 23:35:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             51W /  400W |       5MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# Import the libraries and modules used for fine-tuning a large language model (LLM).

In [17]:
# Access OS utilities, system commands and process management tools
import os, sys, shutil, subprocess

# Work with filesystem paths in a cross-platform, object-oriented way
from pathlib import Path

# Google drive mounting capability
from google.colab import drive

# Date time from system
from datetime import datetime

# Load and manage training datasets from the Hugging Face Hub
from datasets import load_dataset

# Import model, tokenizer, and training utilities from the Transformers library
from transformers import (
    AutoModelForCausalLM,  # Loads a causal language model (e.g., Llama, GPT)
    AutoTokenizer,  # Handles text tokenization for the model
    HfArgumentParser,  # Parses command-line or script arguments
    TrainingArguments,  # Defines training configurations and hyperparameters
    pipeline,  # Creates ready-to-use NLP pipelines for inference
    logging,  # Controls logging verbosity and output
)

# Parameter-Efficient Fine-Tuning (PEFT)
# Give the ability to adapt LLMs without having to redo all the weights.
# Utilizes Low-Rank Adapter (LoRA).
# Efficiently fine-tune without redoing all their parameters.
# Overall reduces compute cost while preserving performance of the model.
from peft import LoraConfig, PeftModel, get_peft_model

# Transformers Reinforcement Learning (TRL)
# Supervised Fine-Tuning (SFT)
# Import the SFTTrainer to manage supervised fine-tuning
from trl import SFTTrainer

# Define and manage training configuration settings (e.g., epochs, batch size, learning rate)
from transformers import TrainingArguments

# Import model and tokenizer classes for causal language modeling
from transformers import AutoModelForCausalLM, AutoTokenizer

# Import Hugging Face pipeline for streamlined text generation tasks
from transformers import pipeline

# Import login utility to authenticate with the Hugging Face Hub
from huggingface_hub import login

# Securely prompt for the API key without showing it in the terminal
import getpass

# Environment Variables

In [9]:
# If your notebook sometimes hides devices, force the first GPU visible:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Disable Weights & Biases tracking to prevent automatic logging
os.environ["WANDB_DISABLED"] = "true"

# Hugging Face Authentication

In [10]:
# Prompt the user to enter their Hugging Face API key securely (hidden input)
hugging_face_key = getpass.getpass("Enter your Hugging Face token: ")

# Log in to Hugging Face Hub using the entered key
login(hugging_face_key)

Enter your Hugging Face token: ··········


# Dataset for fine-tuning

In [11]:
# Dataset for fine-tuning
dataset_name = "nikhiljatiwal/minipython-Alpaca-14k"

dataset = load_dataset(dataset_name)

# Get the dataset structure and size
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/387 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/25.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 14000
    })
})


# Model input and output config.

In [ ]:
# Hugging Face model to train
model_name = "NousResearch/llama-2-7b-chat-hf"  # Optimized for the chat bot.

# Output model name
new_model = "/kaggle/working/llama-2-7b-codeAlpaca"

# Quantized + LoRA = QLoRA which is a method of fine-tuning LLMs more cheaply and efficiently without sacrificing accuracy.

In [ ]:
# LoRA attention dimension (rank)
# The higher the value the more fine-grained updates at the cost of increased
# memory usage
lora_r = 64

# LoRA alpha (scaling factor)
# How much the LoRA layes change the original LLM's behavior.
# Higher value = updates stronger : Lower value = keeps changes smaller.
lora_alpha = 16

# LoRA dropout probability
# Randomly turns off some connections during training to avoid overfitting.
lora_dropout = 0.1

# Training Arguments parameters

In [14]:
# Output directory where the model predictions and checkpoints will be stored
output_dir = "/kaggle/working/llama-2-7b-codeAlpaca"

# Number of training epochs
num_train_epochs = 1

# Enable fp16 training (set to True for mixed precision training)
fp16 = True

# Batch size per GPU for training
per_device_train_batch_size = 8

# Batch size per GPU for evaluation
per_device_eval_batch_size = 8

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 2

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient norm (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "adamw_torch"

# Learning rate schedule
lr_scheduler_type = "constant"

# Group sequences into batches with the same length
# Saves memory and speeds up training considerably
group_by_length = True

# Ratio of steps for a linear warmup
warmup_ratio = 0.03

# Save checkpoint every X updates steps
save_steps = 100

# Log every X updates steps
logging_steps = 10

# Supervised Fine-Tuning (SFT) Model and Tokenizer Setup

In [15]:
# Set max token length for each training example
max_seq_length = None

# Combine shorter samples for faster training
packing = False

# Load the training dataset
dataset = load_dataset(dataset_name, split="train")

# Load and configure the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load the base model in 8-bit precision for memory efficiency
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Enable gradient checkpointing and input gradients for fine-tuning
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

# LoRA Configuration, Dataset Tokenization and Trainer Initialization

In [18]:
# Configure and apply LoRA to the base model
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)

# Define training arguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    max_grad_norm=max_grad_norm,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type=lr_scheduler_type,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    fp16=fp16,
    group_by_length=True,
)


# Tokenize dataset and add padding for consistent lengths.
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512,  # limit sequence length
    )


tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Initialize the supervised fine-tuning trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_arguments,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14000 [00:00<?, ? examples/s]

# Model Training

In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained(new_model)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.597500
20,0.971200
30,0.898600
40,0.804500
50,0.846100
60,0.782300
70,0.752000
80,0.754100
90,0.712500
100,0.769600


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
